# BanglaPhonologyBench — M4 Probing Harness

Extracts hidden-state embeddings for the Task 1 (G2P) and Task 2a (syllable
count) words across 5 tokenizers, then trains linear probes comparing the
A / M / CB tokenization-alignment categories (Spec C.4). See
`docs/probing_integration.md` and `docs/DEVELOPMENT_LOG.md` in the repo for
the full background and the design decisions behind this notebook.

## Resumability — read this first

Each (tokenizer, category) combo is at most ~3,000 words, a single forward
pass per word — minutes, not hours, even for an 8B model on a T4. So the
actual restart risk here isn't "a run dies mid-file," it's "you come back
in a later session (a new week's 30h GPU quota, a disconnect between
combos) and don't want to redo already-finished combos." This notebook
therefore uses **file-level** resumability only: a combo is skipped
entirely if its output already exists and looks complete
(`kaggle_probing_lib.extraction_done`). No row-level checkpointing, no
partial-state files — simpler, and matches the actual scale of the problem.

**To make progress persist across Kaggle sessions:**
1. Run this notebook normally. Checkpoints land in `/kaggle/working/checkpoints/`.
2. When you're done for the day (or hit your GPU quota), click
   **Save Version** (Save & Run All, or Quick Save) — this saves
   `/kaggle/working/` as a new version of *this notebook's own output*,
   which Kaggle lets you attach as a Dataset input to a later session.
3. Next session: **Add Input → Notebook Output Files → (this notebook,
   previous version)**, or promote it to a proper Kaggle Dataset via the
   "New Dataset" button on the output tab. Either way it lands under
   `/kaggle/input/<slug>/`. Set `CHECKPOINT_INPUT` below to that path.
4. Re-run the notebook — already-finished (tokenizer, category) combos
   print "already done, skipping" instead of re-extracting.

**Recommended order:** run `banglat5` first (smallest, ~580M, encoder-only
— a good pipeline smoke test that doesn't burn much GPU quota) before
`tigerllm`/`llama3` (9B/8B, need 4-bit quantization to fit a T4's 16GB).


## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" sentencepiece scikit-learn scipy


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/LihanCanCode/BanglaPhonologyBench.git"
REPO_DIR = "/kaggle/working/BanglaPhonologyBench"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

from kaggle_probing_lib import extract_g2p, extract_rhyme, extraction_done, run_probe_battery
from src.tokenizer_adapter import TOKENIZER_SPECS

print("repo ready:", REPO_DIR)


In [ ]:
# HF token: prefer Kaggle Secrets (Add-ons -> Secrets -> add HF_TOKEN),
# fall back to an environment variable if you've set one another way.
# Needed for meta-llama/Llama-3.1-8B-Instruct (gated); everything else
# works without it (llama3 also has an ungated mirror fallback, see
# TOKENIZER_SPECS, so this is optional even for that one).
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("HF_TOKEN set:", bool(HF_TOKEN))


## 2. Checkpoint restore

Set `CHECKPOINT_INPUT` to a previous run's attached dataset path once you
have one (see the resumability instructions above). Leave as-is for a
first run — there's nothing to restore yet.


In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# CHANGE THIS after your first "Save Version" + re-attach, to resume:
CHECKPOINT_INPUT = "/kaggle/input/banglaphonologybench-m4-checkpoints"

if os.path.isdir(CHECKPOINT_INPUT):
    subprocess.run(f"cp -r {CHECKPOINT_INPUT}/. {CKPT_DIR}/", shell=True, check=False)
    print(f"restored checkpoints from {CHECKPOINT_INPUT}")
else:
    print("no checkpoint input attached -- starting fresh "
          "(this is expected on your first run)")


## 3. Config

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EXPORT_DIR = os.path.join(REPO_DIR, "data", "probing_export")
EMB_DIR = os.path.join(CKPT_DIR, "embeddings")
RESULTS_DIR = os.path.join(CKPT_DIR, "probe_results")
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

CATEGORIES = ["A", "M", "CB"]

# smallest/most-informative first -- see the resumability note above for why
RUN_ORDER = ["banglat5", "tigerllm", "gpt2", "byt5", "llama3"]

T5_FAMILY = {"byt5", "banglat5"}          # encoder-only forward pass
NEEDS_4BIT = {"llama3", "tigerllm"}       # 8B/9B: won't fit a T4 in fp16


## 4. Model loading + extraction

`load_model(llm_key)` returns a ready `extract_fn(word) -> List[np.ndarray]`
(one array per layer, layer 0 = embedding layer) plus the layer count.
Wrapped in try/except in the main loop so one model failing (OOM, no
access, network hiccup) doesn't kill the whole run.

**Note (found on the first real GPU run):** T5-family models (ByT5,
BanglaT5) load in **float32**, not float16 — T5 was trained in
bfloat16/fp32, and plain float16 inference is a well-documented way to get
`inf` in its hidden states. `kaggle_probing_lib` also now detects and
auto-redoes any checkpoint that already contains non-finite values (from
before this fix), so you don't need to manually delete anything — just
`git pull` (cell 3 already does this) and re-run.


In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, T5EncoderModel)


def load_model(llm_key):
    repo_candidates = TOKENIZER_SPECS[llm_key]
    last_err = None
    for repo_id in repo_candidates:
        try:
            tok = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
            if llm_key in T5_FAMILY:
                # ALWAYS fp32 for T5, regardless of device: T5 was trained in
                # bfloat16/fp32, and plain float16 inference is known to
                # overflow internal activations (produces inf hidden states).
                # These are small models (byt5-small, banglat5 ~580M) so
                # fp32 doesn't strain a T4's VRAM.
                model = T5EncoderModel.from_pretrained(
                    repo_id, token=HF_TOKEN, torch_dtype=torch.float32,
                ).to(device)
            elif llm_key in NEEDS_4BIT and device == "cuda":
                bnb = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16)
                model = AutoModelForCausalLM.from_pretrained(
                    repo_id, token=HF_TOKEN, quantization_config=bnb,
                    device_map="auto")
            else:
                model = AutoModelForCausalLM.from_pretrained(
                    repo_id, token=HF_TOKEN,
                    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                ).to(device)
            model.eval()
            print(f"  loaded {llm_key} from {repo_id}")
            return tok, model
        except Exception as e:
            last_err = e
            print(f"  {repo_id} failed ({e}), trying next candidate if any")
    raise RuntimeError(f"could not load any repo for {llm_key}: {last_err}")


def make_extract_fn(tok, model, is_t5):
    def extract_fn(word):
        inputs = tok(word, return_tensors="pt").to(model.device)
        with torch.no_grad():
            if is_t5:
                outputs = model(input_ids=inputs["input_ids"],
                                attention_mask=inputs["attention_mask"],
                                output_hidden_states=True)
            else:
                outputs = model(**inputs, output_hidden_states=True)
        return [h[0, -1, :].float().cpu().numpy() for h in outputs.hidden_states]
    return extract_fn


def load_g2p_csv(path):
    import ast
    import pandas as pd
    df = pd.read_csv(path)
    rows = []
    for _, r in df.iterrows():
        rows.append({"word": r["word"], "phon_vec": ast.literal_eval(r["phon_vec"]),
                    "syllables": ast.literal_eval(r["syllables"])})
    return rows


def load_rhyme_csv(path):
    import pandas as pd
    df = pd.read_csv(path)
    rows = []
    for _, r in df.iterrows():
        rows.append({"word1": r["word1"], "word2": r["word2"], "label": int(r["label"])})
    return rows


## 5. Run extraction

Safe to re-run this cell as many times as you like (across sessions too,
with a restored checkpoint) — already-finished combos are skipped without
loading the model at all costing nothing but the `extraction_done` check.


In [ ]:
for llm_key in RUN_ORDER:
    print(f"\n=== {llm_key} ===")

    csv_paths = {cat: os.path.join(EXPORT_DIR, f"g2p_{llm_key}_{cat}.csv")
                for cat in CATEGORIES}
    out_dirs = {cat: os.path.join(EMB_DIR, f"g2p_{llm_key}_{cat}") for cat in CATEGORIES}

    # quick pre-check without loading the model: anything to do at all?
    rows_by_cat = {}
    for cat in CATEGORIES:
        if os.path.exists(csv_paths[cat]):
            rows_by_cat[cat] = load_g2p_csv(csv_paths[cat])
        else:
            rows_by_cat[cat] = []

    # we don't know n_layers before loading the model once; use a generous
    # upper bound just for the pre-check (real check happens per-combo below)
    already_done_guess = all(
        extraction_done(out_dirs[cat], 1) or not rows_by_cat[cat]
        for cat in CATEGORIES
    )
    if already_done_guess and any(rows_by_cat.values()):
        print("  all categories already extracted (or empty) -- skipping model load")
        continue

    try:
        tok, model = load_model(llm_key)
    except Exception as e:
        print(f"  SKIPPING {llm_key}: could not load ({e})")
        continue

    is_t5 = llm_key in T5_FAMILY
    extract_fn = make_extract_fn(tok, model, is_t5)
    n_layers = len(extract_fn("test"))
    print(f"  n_layers (incl. embedding layer) = {n_layers}")

    for cat in CATEGORIES:
        rows = rows_by_cat[cat]
        out_dir = out_dirs[cat]
        if not rows:
            print(f"  {cat}: 0 words for this tokenizer, skipping")
            continue
        if extraction_done(out_dir, n_layers):
            print(f"  {cat}: already done ({len(rows)} words), skipping")
            continue
        print(f"  {cat}: extracting {len(rows)} words x {n_layers} layers ...")
        extract_g2p(extract_fn, rows, out_dir, n_layers)
        print(f"  {cat}: done")

    del model, tok
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nextraction pass complete.")


## 5b. Run rhyme (Task 3a) extraction

Same file-level-resumable pattern as the G2P extraction above, but for the
rhyme-pair prompt `f"{word1} {word2}"` (matches upstream's rhyme-probe
format) and split by A/M/CB via `scripts/export_for_probing.py`'s
`categorize_pair` join (worse-of-the-two-words' category per pair) —
previously flagged as "not yet built" in this notebook's final cell, now
implemented. Loads each model a second time (simplicity over micro-
optimizing away a second load — consistent with this notebook's existing
"file-level resumability is enough at this scale" reasoning); already-
extracted combos are skipped before any model load, same as section 5.


In [ ]:
for llm_key in RUN_ORDER:
    print(f"\n=== rhyme extraction: {llm_key} ===")

    csv_paths = {cat: os.path.join(EXPORT_DIR, f"rhyme_{llm_key}_{cat}.csv")
                for cat in CATEGORIES}
    out_dirs = {cat: os.path.join(EMB_DIR, f"rhyme_{llm_key}_{cat}") for cat in CATEGORIES}

    rows_by_cat = {}
    for cat in CATEGORIES:
        if os.path.exists(csv_paths[cat]):
            rows_by_cat[cat] = load_rhyme_csv(csv_paths[cat])
        else:
            rows_by_cat[cat] = []

    already_done_guess = all(
        extraction_done(out_dirs[cat], 1) or not rows_by_cat[cat]
        for cat in CATEGORIES
    )
    if already_done_guess and any(rows_by_cat.values()):
        print("  all categories already extracted (or empty) -- skipping model load")
        continue

    try:
        tok, model = load_model(llm_key)
    except Exception as e:
        print(f"  SKIPPING {llm_key}: could not load ({e})")
        continue

    is_t5 = llm_key in T5_FAMILY
    extract_fn = make_extract_fn(tok, model, is_t5)
    n_layers = len(extract_fn("test"))
    print(f"  n_layers (incl. embedding layer) = {n_layers}")

    for cat in CATEGORIES:
        rows = rows_by_cat[cat]
        out_dir = out_dirs[cat]
        if not rows:
            print(f"  {cat}: 0 pairs for this tokenizer, skipping")
            continue
        if extraction_done(out_dir, n_layers):
            print(f"  {cat}: already done ({len(rows)} pairs), skipping")
            continue
        print(f"  {cat}: extracting {len(rows)} pairs x {n_layers} layers ...")
        extract_rhyme(extract_fn, rows, out_dir, n_layers)
        print(f"  {cat}: done")

    del model, tok
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nrhyme extraction pass complete.")


## 6. Train probes (G2P + syllable count)

Reuses whatever categories got extracted for each tokenizer (2-way if only
A and CB exist, e.g. for the English-centric tokenizers where M is
usually empty; full 3-way A/M/CB for TigerLLM and BanglaT5). One results
JSON per tokenizer per task; skipped entirely if it already exists.


In [ ]:
import glob
import json
import pickle

import numpy as np


def count_layers(out_dir):
    return len(glob.glob(os.path.join(out_dir, "layer_*.pkl")))


def run_probes_for_llm(llm_key, target_key, probe_type, tag, control_label=False):
    """target_key: 'phon_vecs' or 'syllable_count' (keys inside each layer_N.pkl)."""
    suffix = "_control_label" if control_label else ""
    results_path = os.path.join(RESULTS_DIR, f"probe_{tag}_{llm_key}{suffix}.json")
    if os.path.exists(results_path):
        print(f"{llm_key}/{tag}{suffix}: already done, skipping")
        return

    emb_dirs = {cat: os.path.join(EMB_DIR, f"g2p_{llm_key}_{cat}") for cat in CATEGORIES}
    available = {cat: d for cat, d in emb_dirs.items() if count_layers(d) > 0}
    if len(available) < 2:
        print(f"{llm_key}/{tag}{suffix}: fewer than 2 categories extracted, skipping")
        return

    n_layers = count_layers(next(iter(available.values())))
    per_layer = {}
    for li in range(n_layers):
        X_by_cat, y_by_cat = {}, {}
        for cat, d in available.items():
            with open(os.path.join(d, f"layer_{li}.pkl"), "rb") as f:
                data = pickle.load(f)
            X_by_cat[cat] = np.array(data["embeddings"], dtype=float)
            y_by_cat[cat] = np.array(data[target_key], dtype=float)
        res = run_probe_battery(X_by_cat, y_by_cat, probe=probe_type, n_seeds=10,
                                control_label=control_label)
        per_layer[f"layer_{li}"] = res
        summary = ", ".join(f"{c}={res['mean'][c]:.3f}" for c in res["mean"])
        print(f"  {tag}{suffix} layer {li}: {summary}")

    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(per_layer, f, indent=2)
    print(f"{llm_key}/{tag}{suffix}: saved -> {results_path}")


for llm_key in RUN_ORDER:
    print(f"\n=== probing {llm_key} ===")
    run_probes_for_llm(llm_key, "phon_vecs", "ridge", "g2p")
    run_probes_for_llm(llm_key, "syllable_count", "ridge", "syllables")
    # random-embedding-preserving, random-label control (Spec A.6)
    run_probes_for_llm(llm_key, "phon_vecs", "ridge", "g2p", control_label=True)
    run_probes_for_llm(llm_key, "syllable_count", "ridge", "syllables", control_label=True)

print("\nprobing pass complete.")


## 6b. Train rhyme probe (Task 3a)

Same `run_probe_battery` machinery, `probe="logistic"` (binary rhyme/
not-rhyme classification) instead of `"ridge"`. One results JSON per
tokenizer, named `probe_rhyme_{llm}.json` so section 7's summary loop
(which globs `probe_*.json` and splits the filename into `tag, llm_key`)
picks it up automatically — no changes needed there for the base scores.


In [ ]:
def run_rhyme_probes_for_llm(llm_key, control_label=False):
    suffix = "_control_label" if control_label else ""
    results_path = os.path.join(RESULTS_DIR, f"probe_rhyme_{llm_key}{suffix}.json")
    if os.path.exists(results_path):
        print(f"{llm_key}/rhyme{suffix}: already done, skipping")
        return

    emb_dirs = {cat: os.path.join(EMB_DIR, f"rhyme_{llm_key}_{cat}") for cat in CATEGORIES}
    available = {cat: d for cat, d in emb_dirs.items() if count_layers(d) > 0}
    if len(available) < 2:
        print(f"{llm_key}/rhyme{suffix}: fewer than 2 categories extracted, skipping")
        return

    n_layers = count_layers(next(iter(available.values())))
    per_layer = {}
    for li in range(n_layers):
        X_by_cat, y_by_cat = {}, {}
        for cat, d in available.items():
            with open(os.path.join(d, f"layer_{li}.pkl"), "rb") as f:
                data = pickle.load(f)
            X_by_cat[cat] = np.array(data["embeddings"], dtype=float)
            y_by_cat[cat] = np.array(data["label"], dtype=float)
        res = run_probe_battery(X_by_cat, y_by_cat, probe="logistic", n_seeds=10,
                                control_label=control_label)
        per_layer[f"layer_{li}"] = res
        summary = ", ".join(f"{c}={res['mean'][c]:.3f}" for c in res["mean"])
        print(f"  rhyme{suffix} layer {li}: {summary}")

    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(per_layer, f, indent=2)
    print(f"{llm_key}/rhyme{suffix}: saved -> {results_path}")


for llm_key in RUN_ORDER:
    print(f"\n=== probing rhyme: {llm_key} ===")
    run_rhyme_probes_for_llm(llm_key)
    run_rhyme_probes_for_llm(llm_key, control_label=True)

print("\nrhyme probing pass complete.")


## 7. Summary table

Aggregates every saved `probe_*.json` (g2p, syllables, and rhyme alike)
into one score table: tokenizer, task, layer, category, mean score, std —
plus a second table of the pairwise one-sided p-values (A > M > CB, Spec
C.6.2), which `run_probe_battery` already computes per layer but which
previously stayed stranded inside the raw per-tokenizer JSON. Both are
saved into the checkpoint dir so they survive a Save Version.


In [ ]:
import pandas as pd

rows = []
pairwise_rows = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "probe_*.json"))):
    fname = os.path.basename(path)[len("probe_"):-len(".json")]
    control = fname.endswith("_control_label")
    if control:
        fname = fname[: -len("_control_label")]
    tag, llm_key = fname.split("_", 1)
    with open(path, encoding="utf-8") as f:
        per_layer = json.load(f)
    for layer_name, res in per_layer.items():
        layer_idx = int(layer_name.split("_")[1])
        for cat, mean_score in res["mean"].items():
            rows.append({
                "llm": llm_key, "task": tag, "layer": layer_idx, "category": cat,
                "control_label": control, "mean_score": mean_score,
                "std_score": res["std"][cat],
            })
        for comparison, p_value in res.get("pairwise_one_sided_p", {}).items():
            pairwise_rows.append({
                "llm": llm_key, "task": tag, "layer": layer_idx,
                "control_label": control, "comparison": comparison, "p_value": p_value,
            })

summary = pd.DataFrame(rows)
summary_path = os.path.join(CKPT_DIR, "probe_results_summary.csv")
summary.to_csv(summary_path, index=False)
print(f"wrote {len(summary)} rows -> {summary_path}")

pairwise_summary = pd.DataFrame(pairwise_rows)
pairwise_summary_path = os.path.join(CKPT_DIR, "probe_pairwise_summary.csv")
pairwise_summary.to_csv(pairwise_summary_path, index=False)
print(f"wrote {len(pairwise_summary)} rows -> {pairwise_summary_path}")

summary.sort_values(["llm", "task", "layer", "category"]).head(30)


## 8. Save your progress

**Click "Save Version" now** (top right — choose "Save & Run All" or
"Quick Save") so `/kaggle/working/checkpoints/` persists as this
notebook's output. Next session, attach that output (or promote it to a
Dataset) as an input, set `CHECKPOINT_INPUT` in cell 2 above to match, and
re-run — finished combos will be skipped automatically.

## Status of previously-flagged gaps

- **Rhyme (Task 3a) probing**: implemented (sections 5b/6b) —
  `scripts/export_for_probing.py`'s `categorize_pair` joins each pair's
  two per-word GTAD/STAD categories into one per-pair category (the worse
  of the two), producing `data/probing_export/rhyme_{tokenizer}_{A,M,CB}.csv`.
  Extraction and probing follow the exact same file-level-resumable /
  `run_probe_battery` pattern as G2P, just with `probe="logistic"` and a
  joint `f"{word1} {word2}"` prompt.
- **One-sided t-tests across the probing battery for the paper's Table 3
  equivalent** (Spec C.6.2): now flattened into `probe_pairwise_summary.csv`
  by section 7, alongside the score table — no more manual aggregation
  needed once real results exist.
- Zero-shot evaluation (M5) is a separate notebook/pipeline entirely —
  this one only does the probing (hidden-state linear-probe) side of M4.
